# Phase 1 — Lesson 03: Matrix Transformations
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 03: Matrix Transformations
**Type:** Build | **Languages:** Python, Julia | **Time:** ~75 minutes
**Prerequisites:** Phase 1, Lessons 01-02 (Linear Algebra Intuition, Vectors & Matrices Operations)



# What You'll Learn

---

## Learning Objectives

- Construct rotation, scaling, shearing, and reflection matrices and apply them to 2D and 3D points
- Compose multiple transformations by matrix multiplication and verify that order matters
- Compute eigenvalues and eigenvectors of 2x2 matrices from the characteristic equation
- Explain why eigenvalues determine PCA directions, RNN stability, and spectral clustering behavior


# The Problem

---

You read about PCA and see "find the eigenvectors of the covariance matrix." You read about model stability and see "check if all eigenvalues have magnitude less than 1." You read about data augmentation and see "apply a random rotation." None of this makes sense until you understand what matrices do to space geometrically.

**Matrices are not just grids of numbers. They are spatial machines.** A rotation matrix spins points. A scaling matrix stretches them. A shearing matrix tilts them. Every transformation a neural network applies to data is one of these operations or a composition of them.


# Transformations as Matrices

---

Every linear transformation in 2D can be written as a 2x2 matrix. The matrix tells you exactly where the basis vectors `[1, 0]` and `[0, 1]` end up. Everything else follows.

```
Standard Basis         Matrix M             After Transformation M
e1 = [1, 0]  --->    M = columns are      e1' = new x-basis
e2 = [0, 1]  --->    new basis vectors    e2' = new y-basis
```

The columns of the matrix show where each basis vector lands. Know that, and you know where every point goes (by linearity).


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport mathdef rotation_2d(theta):    c, s = math.cos(theta), math.sin(theta)    return np.array([[c, -s], [s, c]])# Plot a unit square and its 90° rotationsquare = np.array([[0, 1, 1, 0], [0, 0, 1, 1]])R90 = rotation_2d(math.pi / 2)rotated = R90 @ squarefig, axes = plt.subplots(1, 2, figsize=(10, 5))for ax, pts, label, color in [(axes[0], square, 'Original', '#2c3e50'),                               (axes[1], rotated, 'Rotated 90°', '#e74c3c')]:    closed = np.vstack([pts, pts[0]])    ax.plot(closed[0], closed[1], 'o-', color=color, lw=2, label=label)    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)    ax.set_title(label, fontweight='bold')    ax.legend(fontsize=10); ax.grid(True, alpha=0.3); ax.set_aspect('equal')fig.suptitle('Rotation Preserves Distances and Angles', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()point = np.array([1.0, 0.0])rotated_point = R90 @ pointprint(f"Rotate (1,0) by 90°: ({rotated_point[0]:.1f}, {rotated_point[1]:.1f})")print("Expected: (0, 1)")

# Scaling

---

Scaling stretches or compresses along each axis independently.

```
Before Scaling          Scale sx=2, sy=0.5    After Scaling
A(2, 1)    ----->   S = [[2, 0], [0, 0.5]]   A'(4, 0.5)
B(0, 2)    ----->                                B'(0, 1)
```

Scaling matrix: `S = [[sx, 0], [0, sy]]`


In [ ]:
import numpy as np, mathdef scaling_2d(sx, sy):    return np.array([[sx, 0], [0, sy]])# Plot scaling transformationssquare = np.array([[0, 1, 1, 0], [0, 0, 1, 1]])fig, axes = plt.subplots(1, 3, figsize=(14, 4))transforms = [    (np.eye(2), 'Original', '#2c3e50'),    (scaling_2d(2, 0.5), 'Scale (2×, 0.5×)', '#e74c3c'),    (scaling_2d(0.5, 2), 'Scale (0.5×, 2×)', '#27ae60'),]for ax, (M, label, color) in zip(axes, transforms):    t = M @ square    closed = np.vstack([t, t[0]])    ax.plot(closed[0], closed[1], 'o-', color=color, lw=2, label=label)    ax.set_xlim(-1, 2.5); ax.set_ylim(-1, 2.5)    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)    ax.set_title(label, fontweight='bold')    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_aspect('equal')fig.suptitle('Scaling Stretches Along Each Axis Independently', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Shearing

---

Shearing tilts one axis while keeping the other fixed. It turns rectangles into parallelograms.

```
Before Shear            Shear in x, k=1       After Shear
A(1, 0)    ----->   Shx = [[1, k], [0, 1]]   A(1, 0) unchanged
B(0, 1)    ----->                                B'(1, 1) shifted
```

Shear matrices:
- `Shx = [[1, k], [0, 1]]` shifts x by k · y
- `Shy = [[1, 0], [k, 1]]` shifts y by k · x


In [ ]:
import numpy as np, mathdef shearing_2d(kx, ky):    return np.array([[1, kx], [ky, 1]])# Plot shearing transformationssquare = np.array([[0, 1, 1, 0], [0, 0, 1, 1]])fig, axes = plt.subplots(1, 3, figsize=(14, 4))transforms = [    (np.eye(2), 'Original', '#2c3e50'),    (shearing_2d(1, 0), 'Shear x, k=1', '#e74c3c'),    (shearing_2d(0, 1), 'Shear y, k=1', '#27ae60'),]for ax, (M, label, color) in zip(axes, transforms):    t = M @ square    closed = np.vstack([t, t[0]])    ax.plot(closed[0], closed[1], 'o-', color=color, lw=2, label=label)    ax.set_xlim(-1, 2.5); ax.set_ylim(-1, 2.5)    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)    ax.set_title(label, fontweight='bold')    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_aspect('equal')fig.suptitle('Shearing Turns Rectangles into Parallelograms', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Reflection

---

Reflection mirrors points across an axis or line.

```
Before Reflection       Reflect across y-axis   After Reflection
A(2, 1)   ----->   R = [[-1, 0], [0, 1]]      A'(-2, 1)
```

Reflection matrices:
- Reflect across y-axis: `[[-1, 0], [0, 1]]`
- Reflect across x-axis: `[[1, 0], [0, -1]]`


In [ ]:
import numpy as np, mathdef reflection_x(): return np.array([[1, 0], [0, -1]])def reflection_y(): return np.array([[-1, 0], [0, 1]])# Plot reflectionspoints = np.array([[2, 1, 0, 0], [1, 1, 2, 0]])  # two pointsfig, axes = plt.subplots(1, 3, figsize=(14, 4))transforms = [    (np.eye(2), 'Original', '#2c3e50'),    (reflection_x(), 'Reflect x-axis', '#e74c3c'),    (reflection_y(), 'Reflect y-axis', '#27ae60'),]for ax, (M, label, color) in zip(axes, transforms):    t = M @ points    for i in range(points.shape[1]):        ax.annotate('', xy=t[:, i], xytext=points[:, i],                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5))    ax.scatter(points[0], points[1], color='#95a5a6', s=50, label='original')    ax.scatter(t[0], t[1], color=color, s=50, label=label, marker='x')    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)    ax.set_title(label, fontweight='bold')    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_aspect('equal')fig.suptitle('Reflection Flips Space Across an Axis', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Composition: Order Matters

---

Applying transformation A then B is the same as multiplying their matrices: `result = B @ A @ point`.
**Order matters.** Rotate then scale gives different results than scale then rotate.

Rotate 90° then Scale (2, 0.5):
```
(1, 0) --Rotate 90--> (0, 1) --Scale--> (0, 0.5)
Composed: S @ R = [[0, -2], [0.5, 0]]
```

Scale (2, 0.5) then Rotate 90°:
```
(1, 0) --Scale--> (2, 0) --Rotate 90--> (0, 2)
Composed: R @ S = [[0, -0.5], [2, 0]]
```

Different results. Matrix multiplication is **not commutative**.


In [ ]:
import numpy as np, mathdef rotation_2d(theta):    c, s = math.cos(theta), math.sin(theta)    return np.array([[c, -s], [s, c]])def scaling_2d(sx, sy):    return np.array([[sx, 0], [0, sy]])R = rotation_2d(math.pi / 2)S = scaling_2d(2, 0.5)rotate_then_scale = S @ Rscale_then_rotate = R @ Spoint = np.array([1.0, 0.0])r1 = rotate_then_scale @ pointr2 = scale_then_rotate @ pointprint(f"Rotate 90 then scale:  ({r1[0]:.2f}, {r1[1]:.2f})")print(f"Scale then rotate 90:  ({r2[0]:.2f}, {r2[1]:.2f})")print(f"Same? {np.allclose(r1, r2)}  (matrix multiplication is not commutative)")# Visualize composition differencefig, axes = plt.subplots(1, 2, figsize=(10, 4))for ax, M, label in [(axes[0], rotate_then_scale, 'Rotate then Scale'),                      (axes[1], scale_then_rotate, 'Scale then Rotate')]:    t = M @ point    ax.quiver(0, 0, point[0], point[1], angles='xy', scale_units='xy', scale=1,              color='#2c3e50', label='original (1,0)', width=0.006)    ax.quiver(0, 0, t[0], t[1], angles='xy', scale_units='xy', scale=1,              color='#e74c3c', label='transformed', width=0.006)    ax.set_xlim(-1, 3); ax.set_ylim(-1, 3)    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)    ax.set_title(label, fontweight='bold')    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_aspect('equal')fig.suptitle('Composition Order Changes the Result', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

# Eigenvalues and Eigenvectors

---

Most vectors change direction when a matrix hits them. **Eigenvectors** are special: the matrix only scales them, never rotates them. The scaling factor is the **eigenvalue**.  

```
A @ v = λ · v

v is the eigenvector (direction that survives)
λ is the eigenvalue (how much it stretches)
```

Example: `A = [[2, 1], [1, 2]]`
- Eigenvector `[1, 1]` with eigenvalue 3:  `A @ [1,1] = [3, 3] = 3 × [1, 1]`
- Eigenvector `[1, -1]` with eigenvalue 1: `A @ [1,-1] = [1, -1] = 1 × [1, -1]`

The matrix stretches space by 3× along `[1, 1]` and keeps `[1, -1]` unchanged. Every other direction is a mix of these two.


In [ ]:
import numpy as npA = np.array([[2, 1], [1, 2]], dtype=float)eigenvalues, eigenvectors = np.linalg.eig(A)print(f"Eigenvalues: {eigenvalues}")print(f"Eigenvectors (columns):\n{eigenvectors}\n")for i in range(len(eigenvalues)):    v = eigenvectors[:, i]    lam = eigenvalues[i]    print(f"  A @ v{i} = {np.round(A @ v, 4)},  λ × v{i} = {np.round(lam * v, 4)}")

In [ ]:
import numpy as np, mathimport matplotlib.pyplot as plt# Visualize eigenvectors: A stretches along eigenvector directionsA = np.array([[2, 1], [1, 2]], dtype=float)eigenvalues, eigenvectors = np.linalg.eig(A)# Create a unit circle of pointstheta = np.linspace(0, 2 * math.pi, 100)circle = np.vstack([np.cos(theta), np.sin(theta)])transformed = A @ circlefig, ax = plt.subplots(figsize=(7, 7))ax.plot(circle[0], circle[1], '--', color='#95a5a6', alpha=0.5, label='unit circle')ax.plot(transformed[0], transformed[1], color='#2c3e50', lw=2, label='A @ circle')for i in range(2):    v = eigenvectors[:, i].real    lam = eigenvalues[i].real    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,              color='#e74c3c' if i == 0 else '#27ae60', width=0.006, label=f'λ{i+1}={lam:.1f}')    ax.quiver(0, 0, lam * v[0], lam * v[1], angles='xy', scale_units='xy', scale=1,              color='#e74c3c' if i == 0 else '#27ae60', width=0.004, alpha=0.5)ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)ax.set_aspect('equal'); ax.set_title('Eigenvectors: Directions A Only Stretches', fontsize=12, fontweight='bold')ax.legend(fontsize=9); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

# Eigendecomposition

---

If a matrix has n linearly independent eigenvectors, it can be decomposed:

```
A = V @ D @ V^(-1)

V = matrix whose columns are eigenvectors
D = diagonal matrix of eigenvalues
V^(-1) = inverse of V
```

This says: rotate into eigenvector coordinates, scale along each axis, rotate back.


# Why Eigenvalues Matter

---

**PCA.** The eigenvectors of the covariance matrix are the principal components. The eigenvalues tell you how much variance each component captures. Sort by eigenvalue, keep the top k, and you have dimensionality reduction.

**Stability.** In recurrent networks and dynamical systems, eigenvalues with magnitude > 1 cause outputs to explode. Magnitude < 1 causes them to vanish. This is the vanishing/exploding gradient problem in one sentence.

**Spectral methods.** Graph neural networks use eigenvalues of the adjacency matrix. Spectral clustering uses eigenvalues of the Laplacian. The eigenvectors reveal the structure of the graph.


# Determinant as Volume Scaling Factor

---

The determinant tells you how much a transformation scales area (2D) or volume (3D):

| Determinant | Meaning |
|-------------|---------|
| det = 1     | Area preserved (rotation) |
| det = 2     | Area doubled |
| det = 0     | Space crushed to lower dimension (singular) |
| det = -1    | Area preserved but orientation flipped (reflection) |

| det(Rotation) | = 1 (always) |
| det(Scale sx, sy) | = sx × sy |
| det(Shear) | = 1 (area preserved) |
| det(Reflection) | = -1 (orientation flipped) |


In [ ]:
import numpy as np, mathdef rotation_2d(theta):    c, s = math.cos(theta), math.sin(theta)    return np.array([[c, -s], [s, c]])def scaling_2d(sx, sy):    return np.array([[sx, 0], [0, sy]])def shearing_2d(kx, ky):    return np.array([[1, kx], [ky, 1]])def reflection_y():    return np.array([[-1, 0], [0, 1]])def det_2x2(matrix):    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]print(f"det(rotation 45°) = {det_2x2(rotation_2d(math.pi/4)):.4f}")print(f"det(scale 2,3)    = {det_2x2(scaling_2d(2, 3)):.1f}")print(f"det(shear kx=1)   = {det_2x2(shearing_2d(1, 0)):.1f}")print(f"det(reflect y)    = {det_2x2(reflection_y()):.1f}")singular = np.array([[1, 2], [2, 4]], dtype=float)print(f"det(singular)     = {np.linalg.det(singular):.1f}")print("Singular: columns are proportional, space collapses to a line.")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What do the columns of a transformation matrix represent?
> **Explanation:** Where each basis vector (e1=[1,0], e2=[0,1]) lands after the transformation.

**Q2:** Why does order matter in matrix composition?
> **Explanation:** B @ A means apply A first, then B. Rotate-then-scale ≠ scale-then-rotate; matrix multiplication is not commutative.

**Q3:** What is an eigenvector?
> **Explanation:** A direction that the matrix only scales, never rotates. A @ v = λ · v.

**Q4:** What does the determinant tell you?
> **Explanation:** The factor by which the transformation scales area (2D) or volume (3D). Zero = irreversible.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Apply rotation, scaling, and shearing to a unit square (corners at [0,0], [1,0], [1,1], [0,1]). Print the transformed corners for each. Verify rotation preserves distances between corners.

**Exercise 2:** Find the eigenvalues of [[4, 2], [1, 3]] by hand using the characteristic equation. Verify with your from-scratch function and NumPy.

**Exercise 3:** Create a composition of three transformations (rotate 30°, scale [1.5, 0.8], shear kx=0.3) and apply to 8 points on a circle. Compute the determinant of the composed matrix and verify it equals the product of individual determinants.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Rotation matrix | "Spins things" | An orthogonal matrix that moves points along circular arcs while preserving distances and angles. Determinant is always 1. |
| Scaling matrix | "Makes things bigger" | A diagonal matrix that stretches or compresses independently along each axis. Determinant is the product of scale factors. |
| Shearing matrix | "Slants things" | A matrix that shifts one coordinate proportionally to another, turning rectangles into parallelograms. Determinant is 1. |
| Reflection | "Mirrors things" | A matrix that flips space across an axis or plane. Determinant is -1. |
| Composition | "Do two things" | Multiplying transformation matrices to chain operations. Order matters: B @ A means apply A first, then B. |
| Eigenvector | "Special direction" | A direction that the matrix only scales, never rotates. The transformation's fingerprint. |
| Eigenvalue | "How much it stretches" | The scalar factor by which the matrix scales its eigenvector. Can be negative (flip) or complex (rotation). |
| Eigendecomposition | "Break the matrix apart" | Writing a matrix as V @ D @ V^(-1), separating it into fundamental scaling directions and magnitudes. |
| Determinant | "A single number from a matrix" | The factor by which the transformation scales area (2D) or volume (3D). Zero means irreversible. |
| Characteristic equation | "Where eigenvalues come from" | det(A - λ·I) = 0. The polynomial whose roots are the eigenvalues. |


# Further Reading

---

- [3Blue1Brown: Linear Transformations](https://www.3blue1brown.com/lessons/linear-transformations) — visual intuition for how matrices reshape space
- [3Blue1Brown: Eigenvectors and Eigenvalues](https://www.3blue1brown.com/lessons/eigenvalues) — the best visual explanation of what eigenvectors mean geometrically
- [MIT 18.06 Lecture 21](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/) — Gilbert Strang's classic treatment


# Next Steps

---

When you're ready, continue to Lesson 04: Calculus for Machine Learning.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
